#### **Association Rules and Apriori Algorithm**

**SIMPLE END-TO-END MARKET BASKET RECOMMENDER**

`From a transaction file -> frequent itemset -> rules -> recommendations`
Needs:  `pandas, mlxtend (pip install pandas mlxtend)`

Needs the file: grocery_transactions.csv (in the same folder as this notebook)

In [1]:
import pandas as pd                                     # tables (Dataframes)
from mlxtend.preprocessing import TransactionEncoder    # baskets -> True/False table
from mlxtend.frequent_patterns import apriori           # finds the frequent itemsets
from mlxtend.frequent_patterns import association_rules        # returns itemset into rules
from icecream import ic

In [2]:
# STEP 1 - READ THE TRANSACTION FILE
'''
The file has ONE ROW PER LINE ITEM (not one row per basket). 
Several rows share the same transaction_id: those rows together are one receipt.

'''
file_path = 'grocery_transactions.csv'
df = pd.read_csv(file_path)

print('STEP 1 - file loaded')
print(f'    rows (line items)   : {len(df):,}')
print(f'    transactions        : {df["transaction_id"].nunique():,}') # nunique() = how many distinct values
print(f'    columns             : {list(df.columns)}\n')
 

STEP 1 - file loaded
    rows (line items)   : 13,234
    transactions        : 2,472
    columns             : ['transaction_id', 'timestamp', 'customer_id', 'item', 'category', 'quantity', 'unit_price', 'line_total']



In [3]:
# STEP 2 - ORGANIZE THE DATA: one list of items per transaction

'''
2a) Clean the product names first
- .str.strip() removes spaces at the start and the end ("    butter " -> "butter")
- .str.lower() puts everything in lowercase            ("BUTTER" -> "butter")
without this, "butter", "BUTTER" and " butter " would count as 3 products.
'''
df['item'] = df['item'].str.strip().str.lower()

'''
2b) Group the rows by receipt and collect the item of each receipt.
- gather the item values of each receipt
- run a small function on each group
- remove duplicates inside the same basket (an item scanned twice is still INE item)
- put them in a fixed order (easier to read)

'''

baskets = (df.groupby('transaction_id')['item'].apply(lambda items: sorted(set(items))).tolist() 
           ) # lambda = a one-line function, # .tolist() -> a plain pyhton list 

print('STEP 2 - data organized into baskets')
print(f'    number of baskets: {len(baskets):,}')
print(f'    first 3 baskets     :')
for b in baskets[:7]:
    print('     ', b)
print()


STEP 2 - data organized into baskets
    number of baskets: 2,472
    first 3 baskets     :
      ['eggs', 'parmesan']
      ['baby formula', 'beer', 'chicken breast', 'diapers', 'red wine']
      ['avocado', 'berries', 'granola', 'peanuts', 'whole wheat bread', 'yogurt']
      ['beer', 'berries', 'lettuce', 'tomatoes', 'yogurt']
      ['ground beef', 'orange juice', 'pasta', 'tomato sauce', 'trash bags', 'whole wheat bread']
      ['conditioner', 'salmon', 'shampoo', 'toothpaste', 'whole wheat bread']
      ['eggs', 'ice cream', 'lettuce', 'white bread']



In [4]:
# STEP 3 -  ENCODE THE BASKETS AS A TRUE/FALSE TABLE

'''
Apriori cannot read lists of words; it needs a table with one row per basket, one colum per product, True where the producr is present.

 TransactionEncoder()   : the translator object
 - .fit(baskets)        : LEARN the list of all products (these become the columns)
 - .transform(baskets)  : APPLY it -> the True/False array
 - te.columns_          : the product names it learned, in column order
'''
te = TransactionEncoder()
translated_data = te.fit(baskets).transform(baskets) # We obtain a True/False-like Matrix
basket_table = pd.DataFrame(translated_data, columns=te.columns_)
# basket_table.to_csv('true_false_data.csv')
print('STEP  3 - basket table built')
print(f'    shape: {basket_table.shape}     (baskets x products)\n')
print(basket_table.iloc[:3, :5]) # .iloc[rows, cols] -> first 3 rows, first 5 columns
print()


STEP  3 - basket table built
    shape: (2472, 50)     (baskets x products)

   apples  avocado  baby formula  baby wipes  bacon
0   False    False         False       False  False
1   False    False          True       False  False
2   False     True         False       False  False



In [5]:
# STEP 4 - APRIORI: FIND THE FREQUENT ITEMSETS
'''
apriori() keeps only the combinations of products that appear often enough.

- basket_table          the True/False table from STEP 3
- min_support=0.02      KEEP an itemset only if it appears in at least 2% of baskets.
                        Support is a FRACTION: 0.02 x 2472 baskets = about 50 baskets.
                        Higher value -> fewer, stronger itemsets and a faster run
                        Lower value -> more itemsets, including coincidences.
- use_colnames          show PRODUCT NAMES in the result instead of column numbers.
                        ALWAYS use True: without it you get frozenset({7, 19})
- max_len=3             stop at combinations of 3 products. Longer ones are rare and
                        hard to act on. Set it to None for no limit
                        
'''
frequent_itemsets = apriori(basket_table,
                            min_support=0.02,
                            use_colnames=True,
                            max_len=3
                            )

# The rsult has 2 columns: 'support' {the fraction} and 'itemsets' (the products)
# Add one helper column: how many product are in each itemset

frequent_itemsets['size'] = frequent_itemsets['itemsets'].apply(len) # apply(len) = length of each set

print('STEP 4 - frequent itemsets found by Apriori')
print(f'    total itemsets : {len(frequent_itemsets)}\n')
by_size = frequent_itemsets["size"].value_counts().sort_index()
print(f'    by size        : {by_size.to_dict()}')
print(' \nthe 5 most frequent itemsets of 2 products:')
pairs = frequent_itemsets[frequent_itemsets['size']==2].nlargest(5, 'support') #nlargest = top rows
print()
for _, row in pairs.iterrows():
    print(f'    {row["support"]:.3f}    {set(row["itemsets"])}')
print()
    


# trash = frequent_itemsets
# trash['itemsets'] = trash['itemsets'].apply(lambda x: ', '.join(list(x)))
# trash

STEP 4 - frequent itemsets found by Apriori
    total itemsets : 184

    by size        : {1: 50, 2: 86, 3: 48}
 
the 5 most frequent itemsets of 2 products:

    0.104    {'tomato sauce', 'pasta'}
    0.102    {'white bread', 'butter'}
    0.096    {'milk 2%', 'coffee beans'}
    0.082    {'strawberry jam', 'white bread'}
    0.082    {'parmesan', 'pasta'}



In [6]:
# STEP 5 - GENERATE THE RULES
'''
association_rules() splits every frequent itemset into IF -> THEN in every possible way, and computes the quality measures of each split

frequent_itemsets   the table from STEP 4
metric='lift'       which measure to filter on
min_threshold=1.2   keep a rule only if lift >= 1.2 
lift = confidence/support(Y)
       lift =  1 -> useless (Y is just popular)
       lift > 1  -> real association
       lift < 1  -> the products avoid each other


'''
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)

# -- Keep only the rules we can actually serve -----


'''
1) ONE product on the THE side: we recommend single items, not bundles
rules['support']; in the market basket analysis, "support" is the percentage of all transactions 
that contain the items in the rule (e.g., 0.02 means 2% of baskets)
len(baskets); the total number of shopping carts in our dataset
rules['support'] * len (baskets)): converts the fraction into a raw count of real-world baskets
.round().astype(int) cleans up the math to turn it into a whole number 
.apply(len)==1: keeps rules whose 'consequents' set has exactly one product
 2) confidence >= 0.3: right at least 30% of time
 3) at least 40 real baskets behind the rule, so it is not a coincidence.
       (support is a fraction, so support x number_of_basket = number of baskets)
'''
rules['n_baskets'] = (rules['support'] * len (baskets)).round().astype(int)
rules = rules[(rules['consequents'].apply(len)==1) &  #& means "and" (one condition per bracket)
              (rules['confidence'] >= 0.30) & 
              (rules['n_baskets'] >=40)]

# print(len (baskets))
# rules
# Sort the best rules first: highest lift at the top
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
# ic(rules)

print('STEP 5 - rules generated')
print(f'      Usable rules : {len(rules)}')
print('       The 5 strongest rules:')
for _ , r in rules.head().iterrows():
       # print(f"{r['antecedents']} -> {r['consequents']}")
       left = ', '.join(sorted(r['antecedents'])) # frozenset -> readable text
       right = ', '.join(sorted(r['consequents']))
       print(f"      IF {left:<28} THEN {right:<16} | conf {r['confidence']:.0%}  lift {r['lift']:.2f}")
       



STEP 5 - rules generated
      Usable rules : 245
       The 5 strongest rules:
      IF baby formula, baby wipes     THEN diapers          | conf 83%  lift 7.04
      IF baby formula, diapers        THEN baby wipes       | conf 64%  lift 6.91
      IF baby wipes, diapers          THEN baby formula     | conf 51%  lift 6.22
      IF beer, chocolate bar          THEN tortilla chips   | conf 74%  lift 6.05
      IF beer, salsa                  THEN tortilla chips   | conf 72%  lift 5.89


In [7]:
# STEP 6 - SAVE THE RULES TO A FILE
'''
The rules are the "knowledge" of the recommender. We compute them once (slow)
and save them, so the recommendation part can just read the file.

A frozenset cannot be written to a CSV, so convert both sides to the text
'''

rules_to_save = rules.copy()
rules_to_save['antecedents'] = rules_to_save['antecedents'].apply(lambda s: ', '.join(sorted(s)))
rules_to_save['consequents'] = rules_to_save['consequents'].apply(lambda s: ', '.join(sorted(s)))
rules_to_save

# Keep only useful columns, in a sensible order
rules_to_save = rules_to_save[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]
rules_to_save.to_csv('rules.csv', index=False) # index=False -> do not write the row numbers
rules_to_save.to_json('rules.json', orient="records", indent=4)

print('STEP 6 - rules saved to rules.csv')
print(rules_to_save.head(3).to_string(index=False))

STEP 6 - rules saved to rules.csv
             antecedents  consequents  support  confidence     lift  n_baskets
baby formula, baby wipes      diapers 0.023463    0.828571 7.038586         58
   baby formula, diapers   baby wipes 0.023463    0.637363 6.910353         58
     baby wipes, diapers baby formula 0.023463    0.513274 6.219677         58


In [8]:
# STEP 7 - THE RECOMMENDER

'''
Suggest products for a basket, with an explanation for each suggestion.

Recoomend products based on the customer's existing basket and the association rules
learned from historical shopping transactions.

PARAMETERS:
basket      : a list of products the customer already has, e.g. ['pasta', 'diaper']
how_many:   : how many suggestions to return (default 3). Maximum number of recommendations to return.

RETURNS:
A list of dictionaries, each containing {'item', 'score', 'because'}
- 'item'    -> Recommended product name.
- 'score'   -> List of score rounded to two decimal places.
- 'because' -> Explanation for the recommendation.

REQUIREMENT:
The Dataframe of rules or Json file must already exist.
'''
# Load saved rules and reconstruct product sets for the recommender.
rules = pd.read_csv('rules.csv')
for column in ['antecedents', 'consequents']:
    rules[column] = rules[column].apply(
        lambda text: frozenset(item.strip().lower() for item in text.split(',') if item.strip())
    )
def recommend(basket, how_many=3):
    # STEP 7.1: CONVERT THE BASKET INTO A SET
    '''
    Example: 
    ['pasta', 'ground beef', 'pasta']
    becomes:
    ['pasta', 'ground beef']
    Sets remove duplicated and make membership checks convenient
    '''
    
    basket = set(basket) # a set makes the "is it inside?" test easy
    
    # STEP 7.2: CREATE DICTIONARIES TO STORE RECOMMENDATIONS
    '''
    Dictionary storing each recommended product and its best lift score.add()
    {
        'tomato sauce': 2.4,
        'parmesan': 1.8
    }
    
    '''
    suggestions = {} # product  -> its best score so far
    
    '''
    Dictionary storing the explanation associated with each product.
    
    The product name is the key, and its explanation is the value
    
    Example:
    
    {
        'tomato sauce': 'bacause you bought pasta ...'
    }
    '''
    reason = {}
    
    # STEP 7.3 EXAMINE EVERY ASSOCIATION RULE
    '''
    `rules` is a pandas dataFrame containing the assicaiation rules learned in the previous steps
    `.iterrows()` processes the DataFrame on row at a time.
    `_` receives the row index, which we do not need.
    `rule` contains the current row's data.
    '''
    # Look at every rule we kep in STEP 5
    for _, rule in rules.iterrows():
        # STEP 3A: GET THE ANTECEDENT (THE "IF" PART)
        '''
        Retrieve the products required for this rule to apply.
        Example:
        {'pasta', 'ground beef'}
        
        Convert them into a set for easy comparison with the basket
        '''
        
        if_part = set(rule['antecedents'])
        
        
        # STEP 3B: GET THE CONSEQUENT (THE "THEN" PART)
        '''
        Retrieve the product suggested by the rule.
        `list()` converts the consequent collection into a list
        `[0]` selects the first product
        
        '''
        then_part = list(rule['consequents'])[0]
        # print(if_part, then_part)
        
        # SETP 7.4 CHECK WHETHER THE RULE APPLIES
        '''
        `.issubset(basket)` checks whether EVERY product in `if_part` is present in the customer's basket
        `not` reverses the result
        It even one required product is missing, skip this rule
        '''
        
        if not if_part.issubset(basket):
            continue
        
        # STEP 7.5 DO NOT RECOMMEND PRODCUTS ALREADY IN THE BASKET
        '''
        Check whether the suggested product is already in the basket. 
        Never recommend something the customer already has
        If it is, skip this rule and examine the next one.
        '''
        
        if then_part in basket:
            continue
        
        # STEP 7.6 KEPP THE STRONGEST REULE FOR EACH PRODUCT
        
        '''
        A product may be suggested by multiply rules.
        For example:
        
        pasta                   -> tomato sauce lift = 1.7
        pasta + ground beef     -> tomato sauce lift = 2.4
        
        Keep the rule with the highest lift score
        `or` means the condition is true if the product is new OR 
        the current rule has a higher lift that the previous stored score.
         
        '''
        # if (then_part not in suggestions
        #     or rule['lift'] > suggestions[then_part]
        #     ):
        if then_part not in suggestions or rule['lift'] > suggestions[then_part]:
            
            # Store the strongest lift score found for this product
            suggestions[then_part] = rule['lift']
            '''
            Sort the antecedent products alphabetically and join them into a readable phrase
            `' and '.join(if_part)`
            Example:
            
            {'pasta', 'ground beef'}
            becomes:
            'ground beef and pasta'
            
            '''
            trigger = ' and '.join(if_part)
            
            # Build an explanation using f-string.
            '''
            {trigger} inserts the products that activated the rule.
            {then_part} inserts the recommended product
            
            {rule['confidence']:.0%} formats confidence as a percentage with zero decimal places.
            
            '''
            
            reason[then_part] = (
                f"because you bought {trigger} - "
                f"{rule['confidence']:.0%} of baskets with "
                f"{trigger} also contain {then_part}, "
                f"{rule['lift']:.1f}x more often than average "
                f"(based on {rule['n_baskets']} real baskets)"
            )
            
            # Save the explanation under the recommended product's name.
            # if a stronger rule is found later, this explanation will be replaced along with the score.
    # STEP 7.7 SORT THE RECOMMENDATIONS
    
    # Sort the suggestions from the highest score to the lowest, keep the top ones.
    # sorted(..., key =..., reverse=True) sort by the score, biggest first
    best = sorted(
        suggestions.items(), 
        key=lambda pair: pair[1], 
        reverse=True
        )[:how_many]
    
    '''
    
    Example result:
    
    [
        {
            'item': 'tomato sauce,
            'score': 2.4,
            'because': 'because you bought pasta ..'
        }
    ]
    if no eligible recommendations exist, this returns an empty list
    '''
    
    return [
        {
            'item': item,
            'score': round(score, 2),
            'because': reason[item]
        } for item, score in best
    ]

In [9]:
# STEP 8 - TRY IT

print('STEP 8 - recommendations\n')

test_baskets = [
    ['pasta'],
    ['pasta', 'ground beef'],
    ['beer', 'tortilla chips'],
    ['diapers'],
    ['yogurt', 'berries'],
    ['shampoo']
]

for id, basket in enumerate(test_baskets):
    print(f'BASKET {id + 1}: {", ".join(basket)}')
    results = recommend(basket, how_many=3)
    
    if not results:
        print('     (no rule applies to this basket)')
        
    
    else:
        for r in results:
            print(f'    -> {r["item"]}')
            print(f'    -> {r["because"]}')
        print()
print(f'Done. The rules are in rules.csv - Open it to see everything the model learned.')
            

STEP 8 - recommendations

BASKET 1: pasta
    -> tomato sauce
    -> because you bought pasta - 60% of baskets with pasta also contain tomato sauce, 3.7x more often than average (based on 258 real baskets)
    -> parmesan
    -> because you bought pasta - 47% of baskets with pasta also contain parmesan, 3.4x more often than average (based on 202 real baskets)

BASKET 2: pasta, ground beef
    -> parmesan
    -> because you bought pasta and ground beef - 62% of baskets with pasta and ground beef also contain parmesan, 4.5x more often than average (based on 71 real baskets)
    -> tomato sauce
    -> because you bought pasta and ground beef - 72% of baskets with pasta and ground beef also contain tomato sauce, 4.4x more often than average (based on 82 real baskets)

BASKET 3: beer, tortilla chips
    -> salsa
    -> because you bought beer and tortilla chips - 54% of baskets with beer and tortilla chips also contain salsa, 5.2x more often than average (based on 97 real baskets)
    -> pe

In [10]:
a =  {'pasta', 'ground beef'}

b = ' and '.join(sorted(a))

print(b)

dict = {}
dict['name'] = 'Steve'
dict['age'] = 21
print(dict)

prods = {}
if_part = {'pasta', 'diapers'}
then_part = 'sugar'

prods[then_part] = if_part
print(prods)



ground beef and pasta
{'name': 'Steve', 'age': 21}
{'sugar': {'pasta', 'diapers'}}


In [11]:
for _, rule in rules.iterrows():
    print(list(rule['consequents'])[0])

diapers
baby wipes
baby formula
tortilla chips
tortilla chips
salsa
beer
toilet paper
salsa
tortilla chips
yogurt
yogurt
beer
toilet paper
beer
yogurt
beer
yogurt
salmon
tortilla chips
lemon
peanuts
laundry detergent
yogurt
berries
yogurt
salsa
berries
paper towels
yogurt
berries
granola
bananas
pasta
trash bags
peanuts
granola
berries
bananas
berries
granola
pasta
granola
pasta
bananas
sugar
tomato sauce
pasta
paper towels
tomato sauce
parmesan
parmesan
pasta
pasta
pasta
bananas
white bread
white bread
white bread
parmesan
tomato sauce
white bread
white bread
tomato sauce
strawberry jam
tomato sauce
croissant
strawberry jam
peanuts
parmesan
parmesan
white bread
parmesan
butter
tomato sauce
white bread
diapers
baby wipes
strawberry jam
parmesan
butter
butter
butter
coffee beans
strawberry jam
strawberry jam
chocolate bar
coffee beans
tortilla chips
beer
butter
butter
white bread
avocado
avocado
butter
coffee beans
tomato sauce
strawberry jam
white bread
shampoo
conditioner
yogurt
berri